#**RAG from Scratch with FAISS + OpenAI + Llumo Evaluation**

#**📘 Notebook Overview**
In this notebook, you'll:

- Load a PDF file and extract text using pdfplumber

- Split the text into manageable chunks

- Use OpenAI's text-embedding-ada-002 to embed chunks

- Store them in a FAISS vector store for similarity search

- Perform RAG by retrieving the most relevant chunks and answering using gpt-3.5-turbo

- Evaluate data using the Llumo SDK on metrics like Context Utilization and Hallucination

✨ **Metrics included:**  
- **Context Utilization** ⚖️  
- **Redundancy Reduction** 🔍  
- **Relevance Retention** ☣️  
- **Semantic Cohesion**
- **Hallucination** 🚫  

---


###**📦 1. Install Dependencies**


In [19]:
!pip install openai faiss-cpu pdfplumber llumo -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 4.6 MB/s eta 0:00:00


###**🔐 2. Set API Keys Securely**

In [33]:
import os
from openai import OpenAI

# Set your API keys using environment variables
os.environ["OPENAI_API_KEY"] = "Your Open AI key here"   # 🔁 Replace with your Open AI key
os.environ["LLUMO_API_KEY"] = "Your Llumo key here"   # 🔁 Replace with your Llumo key

# Llumo key (used later)
llumo_key = os.getenv("LLUMO_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

###**📄 3. Load and Preprocess PDF**

In [42]:
import pdfplumber

# Load and extract all text from PDF
def load_pdf_text(file_path):
    all_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            all_text += page.extract_text() + "\n"
    return all_text

pdf_text = load_pdf_text("Howsuccessfulpeoplethink.pdf")  # 📥 Replace with Your PDF file name

###**✂️ 4. Split Text into Chunks**

In [43]:
# Split long text into word-based chunks
def split_text(text, chunk_size=500):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunks = split_text(pdf_text)

###**🔗 5. Generate Embeddings using OpenAI**

In [36]:

# Initialize OpenAI client (v1+)
openaiClient = OpenAI(api_key=openai_key)

# Get vector using text-embedding-ada-002
def get_embedding(text):
    response = openaiClient.embeddings.create(
        input=[text],
        model="text-embedding-ada-002"
    )
    return response.data[0].embedding


###**6. Build FAISS Index**


In [44]:
import faiss
import numpy as np

# Create vector DB with OpenAI embeddings
def build_faiss_index(chunks):
    dim = 1536  # text-embedding-ada-002 output dimension
    index = faiss.IndexFlatL2(dim)
    vectors = [get_embedding(chunk) for chunk in chunks]
    index.add(np.array(vectors).astype("float32"))
    return index, vectors

index, vectors = build_faiss_index(chunks)


###**7. Retrieve Top-k Relevant Chunks**

In [38]:
# Retrieve top-k most similar text chunks for a query
def retrieve_context(query, chunks, index, vectors, k=3):
    query_vec = get_embedding(query)
    distance, indx = index.search(np.array([query_vec]).astype("float32"), k)
    return [chunks[i] for i in indx[0]]


###**🤖 8. Generate Final Answer Using GPT**

In [39]:
# Construct prompt with context and send to ChatGPT
def generate_answer(query, context_chunks):
    context = "\n".join(context_chunks)
    prompt = f"Answer the question using the context below:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

    response = openaiClient.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()


**Ask Questions and Generate Output**

In [40]:
# Define your queries
queries = [
    "What are the benefits of meditation?",
    "Explain the main idea of the document."
]

# Storing input data for eval
results = []
for query in queries:
    context = retrieve_context(query, chunks, index, vectors, k=3)
    answer = generate_answer(query, context)

    results.append({
        "query": query,
        "context": "\n".join(context),
        "output": answer
    })



###**📊Evaluate Output with Llumo SDK**


In [45]:
# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize Llumo client
client = LlumoClient(api_key=llumo_key)

# Evaluate the RAG results
resultDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Context Utilization", "Redundancy Reduction", "Relevance Retention","Semantic Cohesion","Hallucination"], # List of metrics
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
)



Processing Batches: 100%|██████████| 5/5 [00:19<00:00,  3.90s/batch]


In [46]:
resultDf


,query,context,output,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason,Relevance Retention,Relevance Retention Reason,Semantic Cohesion,Semantic Cohesion Reason,Hallucination,Hallucination Reason
0,What are the benefits of meditation?,backyard; or a few hours in a comfortable chai...,The benefits of meditation include increasing ...,2,The response provides information about the be...,52,"The text contains some redundancy, with certai...",1,The context does not contain any information a...,78,"The context maintains a reasonable flow, but s...",58,The output includes information about the bene...
1,Explain the main idea of the document.,"machines, but the great engineer is the man wh...",The main idea of the document is to emphasize ...,75,The response partially utilizes the context by...,56,The context has some redundancy. Some ideas ar...,73,The context provides a good amount of relevant...,76,The context mostly maintains logical flow and ...,20,"The output is mostly supported by the context,..."
